# 04 — Artefak galeri untuk aplikasi

Salinan publik dari alur eksperimen. Tidak memuat dataset, nama peserta, embedding, foto, output cell, atau path komputer penulis. Gunakan dataset sendiri yang berhak diproses. Hasil numerik akan bergantung pada dataset dan tidak harus sama dengan README.

Setelah dijalankan pada data pribadi, output notebook dapat kembali mengandung nama atau wajah. Jalankan script pembersih sebelum commit. Lihat `notebooks_public/README.md` untuk prasyarat dan batas reproduksi.

Selesaikan notebook 02 dan 03 dahulu. Notebook ini membentuk galeri seluruh identitas menggunakan dua referensi dari seed 42. File keluaran disimpan secara lokal dan tetap privat. Notebook tidak membuat tabel Supabase, tidak melakukan migrasi, dan tidak otomatis mengisi database aplikasi. Threshold dari galeri kalibrasi dipakai sebagai nilai provisional.

## Galeri seluruh identitas

In [ ]:
from pathlib import Path
import json
import hashlib

import numpy as np
import pandas as pd

from pathlib import Path
import sys

def find_project_root():
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "requirements.txt").is_file():
            return candidate
    raise FileNotFoundError("Jalankan notebook di dalam checkout repository proyek.")

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
METADATA_DIR = PROJECT_ROOT / "dataset" / "metadata"

embedding_path = ARTIFACTS_DIR / "arcface_embeddings.npz"

with np.load(embedding_path, allow_pickle=False) as saved:
    X = saved["embeddings"]
    image_ids = saved["image_ids"]
    person_ids = saved["person_ids"]

metadata_df = pd.read_csv(
    ARTIFACTS_DIR / "arcface_embedding_metadata.csv"
)

protocol_df = pd.read_csv(
    METADATA_DIR / "fewshot_manifest.csv"
)

gallery_protocol = (
    protocol_df.loc[protocol_df["seed"].eq(42)]
    .sort_values("person_id")
    .reset_index(drop=True)
)

labels = gallery_protocol["person_id"].to_numpy(dtype=str)

if (
    len(labels) != len(np.unique(person_ids))
    or not np.array_equal(labels, np.unique(person_ids))
):
    raise ValueError("Pembagian galeri tidak mencakup seluruh identitas.")

id_lookup = {
    image_id: index
    for index, image_id in enumerate(image_ids)
}

reference_ids = gallery_protocol[
    ["gallery_image_id_1", "gallery_image_id_2"]
].to_numpy(dtype=str)

reference_indices = np.array([
    [id_lookup[value] for value in pair]
    for pair in reference_ids
])

if np.unique(reference_ids).size != len(labels) * 2:
    raise ValueError("Foto referensi galeri terduplikasi.")

if not np.all(
    person_ids[reference_indices] == labels[:, None]
):
    raise ValueError("Foto referensi tidak sesuai identitas.")

centroids = X[reference_indices].mean(axis=1)

lengths = np.linalg.norm(
    centroids,
    axis=1,
    keepdims=True
)

if not np.isfinite(centroids).all() or np.any(lengths <= 1e-12):
    raise ValueError("Centroid tidak valid.")

centroids = (centroids / lengths).astype(np.float32)

identity_names = (
    metadata_df[["person_id", "person_name"]]
    .drop_duplicates()
)

if identity_names["person_id"].duplicated().any():
    raise ValueError("Satu identitas memiliki beberapa nama.")

name_map = identity_names.set_index("person_id")["person_name"]

if not set(labels).issubset(set(name_map.index)):
    raise ValueError("Ada nama identitas yang belum tersedia.")

names = name_map.loc[labels].to_numpy(dtype=str)

gallery_path = ARTIFACTS_DIR / f"gallery_{len(labels)}.npz"

# Label disimpan sebagai string Unicode agar bisa dibaca
# dengan allow_pickle=False.
np.savez_compressed(
    gallery_path,
    centroids=centroids,
    person_ids=labels,
    person_names=names,
    reference_image_ids=reference_ids
)

gallery_protocol.to_csv(
    METADATA_DIR / f"gallery_{len(labels)}_manifest.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Galeri tersimpan:", gallery_path)

## Konfigurasi inferensi

In [ ]:
calibration_config = json.loads(
    (ARTIFACTS_DIR / "open_set_threshold.json")
    .read_text(encoding="utf-8")
)

app_config = {
    "schema_version": 1,
    "method": "centroid",
    "gallery_file": str(gallery_path.relative_to(PROJECT_ROOT).as_posix()),
    "n_identities": len(labels),
    "reference_photos_per_identity": 2,
    "gallery_selection_seed": 42,
    "aligned_image_size": 112,
    "recognition_model": (
        "weights/insightface/models/buffalo_l/w600k_r50.onnx"
    ),
    "detection_model": (
        "weights/insightface/models/buffalo_l/det_10g.onnx"
    ),
    "detector_input_size": [640, 640],
    "detector_threshold": 0.5,
    "recognition_threshold": float(
        calibration_config["threshold"]
    ),
    "recognition_threshold_status": (
        f"provisional_for_{len(labels)}_identities"
    ),
    "threshold_calibration_scope": (
        f"{calibration_config['gallery_identities']} identities, two reference photos per identity"
    ),
    "gallery_sha256": hashlib.sha256(
        gallery_path.read_bytes()
    ).hexdigest(),
    "source_embeddings_sha256": hashlib.sha256(
        embedding_path.read_bytes()
    ).hexdigest(),
}

config_path = ARTIFACTS_DIR / "inference_config.json"

config_path.write_text(
    json.dumps(app_config, indent=2),
    encoding="utf-8"
)

print("Konfigurasi tersimpan:", config_path)

## Pemeriksaan artefak

In [ ]:
with np.load(gallery_path, allow_pickle=False) as saved:
    print("Bentuk centroid :", saved["centroids"].shape)
    print("Jumlah identitas:", len(saved["person_ids"]))
    print("Jumlah nama     :", len(saved["person_names"]))
    print("Bentuk referensi:", saved["reference_image_ids"].shape)
    print(
        "Normalisasi valid:",
        np.allclose(
            np.linalg.norm(saved["centroids"], axis=1),
            1.0,
            atol=1e-5
        )
    )